# Task 2

## Load dataset and TF-IDF features

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.sparse import load_npz

X_full_train_tfidf_v3 = load_npz(f"../preprocessing/full_train_sparse_5000_v3.npz")
X_full_test_tfidf_v3 = load_npz(f"../preprocessing/full_test_sparse_5000_v3.npz")

train_df = pd.read_csv("../Data/train.csv", on_bad_lines='skip', delimiter="\t")
X = train_df.iloc[:, :-1]
y = train_df.iloc[:, -1]

test_df=pd.read_csv("../Data/test.csv", on_bad_lines='skip', delimiter="\t")

In [5]:
print(X_full_train_tfidf_v3.shape)

(139156, 5000)


## Feature Selection

In [ ]:
from scipy.sparse import load_npz
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import metrics

train_label_reference= (pd.read_csv("../Data/train.csv", on_bad_lines='skip', delimiter="\t")["id"]).to_frame(name="id")
test_label_reference=(pd.read_csv("../Data/test.csv", on_bad_lines='skip', delimiter="\t")["id"]).to_frame(name="id")

tfidf_spaces=[2000,1000,500,100]
y_train_df=pd.read_csv("../Data/train.csv", on_bad_lines='skip', delimiter="\t").iloc[:, -1]

def knn_FS(feature_spaces):
    for space in feature_spaces:
        #load the test, train and labels for each feature space
        x_train_df=load_npz(f"../preprocessing/full_train_sparse_{space}_v3.npz")
        x_test_df=load_npz(f"../preprocessing/full_test_sparse_{space}_v3.npz")

        #k nearest neighbour with 2 neighbours
        knn=KNeighborsClassifier(n_neighbors=2)
        knn.fit(x_train_df,y_train_df)

        #predict and send answer to csv
        y_pred=knn.predict(x_test_df)
        print(f"Space: {space} y_pred: {y_pred}")
        test_label_reference["label_id"]=y_pred
        test_label_reference.to_csv(f"Output/FS_{space}.csv",index=False)

ans_FS=knn_FS(tfidf_spaces)


## Dimension Reduction using TruncatedSVD

In [9]:
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

feature_space = [2000, 1000, 500, 100]

for space in feature_space:
    # reduce dimension
    svd = TruncatedSVD(n_components=space)
    X_train_transformed = svd.fit_transform(X_full_train_tfidf_v3)
    X_test_transformed = svd.transform(X_full_test_tfidf_v3)

    # train KNN model
    model = KNeighborsClassifier(n_neighbors=2)
    model.fit(X_train_transformed, y)
    y_pred = model.predict(X_test_transformed)

    print(f"Feature space: {space}, predictions: {y_pred}")
    submission = pd.DataFrame({
        "id": test_df["id"],
        "label_id": y_pred
    })

    submission.to_csv(f"PCA_{space}_predictions_v3.csv", index=False)

Feature space: 2000, predictions: [15  9 23 ...  8 24  0]
Feature space: 1000, predictions: [38 21  4 ...  0  9 11]
Feature space: 500, predictions: [38  4  4 ...  8 30 18]
Feature space: 100, predictions: [38  4  4 ...  0 30  0]


### Using metric="cosine"
This was done to see how adding metric="cosine" to the KNN classifier would affect the predictions.

In [ ]:
# trained on full training set

from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

feature_space = [2000, 1000, 500, 100]

for space in feature_space:
    # reduce dimension
    svd = TruncatedSVD(n_components=space)
    X_train_transformed = svd.fit_transform(X_full_train_tfidf_v3)
    X_test_transformed = svd.transform(X_full_test_tfidf_v3)

    # train KNN model
    model = KNeighborsClassifier(n_neighbors=2, metric="cosine")
    model.fit(X_train_transformed, y)
    y_pred = model.predict(X_test_transformed)

    print(f"Feature space: {space}, predictions: {y_pred}")
    submission = pd.DataFrame({
        "id": test_df["id"],
        "label_id": y_pred
    })

    submission.to_csv(f"PCA_{space}_predictions_v3_cosine.csv", index=False)

Feature space: 2000, predictions: [38  4  4 ...  8 10  0]
Feature space: 1000, predictions: [38  4  4 ... 10 10  0]
Feature space: 500, predictions: [38  4  4 ...  8 10  0]
Feature space: 100, predictions: [38  4  4 ...  0 30  0]


In [4]:
X_train_tfidf_v3 = load_npz(f"../preprocessing/train_sparse_5000_v3.npz")
X_val_tfidf_v3 = load_npz(f"../preprocessing/val_sparse_5000_v3.npz")
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# trained on train-val split

from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

feature_space = [2000, 1000, 500, 100]

for space in feature_space:
    # reduce dimension
    svd = TruncatedSVD(n_components=space)
    X_train_transformed = svd.fit_transform(X_train_tfidf_v3)
    X_val_transformed = svd.transform(X_val_tfidf_v3)

    # train KNN model
    model = KNeighborsClassifier(n_neighbors=2, metric="cosine")
    model.fit(X_train_transformed, y_train)
    y_pred = model.predict(X_val_transformed)

    score = model.score(X_val_transformed, y_val)
    print(f"F1 score for {space}", score)

    print(f"Feature space: {space}, predictions: {y_pred}")
    submission = pd.DataFrame({
        "id": X_val["id"],
        "label_id": y_pred
    })

    submission.to_csv(f"PCA_{space}_val_predictions_v3_cosine.csv", index=False)

F1 score for 2000 0.5227076746191435
Feature space: 2000, predictions: [ 4 26 26 ...  0 14 18]
F1 score for 1000 0.5311871227364185
Feature space: 1000, predictions: [ 4 26 26 ...  0  5 18]
F1 score for 500 0.5411396953147456
Feature space: 500, predictions: [ 4 26 26 ...  0  5 18]
F1 score for 100 0.5558350100603622
Feature space: 100, predictions: [ 4 26 26 ...  5  5 18]
